# ❄️ Snowflake Data Warehouse Analytics & Retention Engine
## CUSTOMER_CHURN_DB Queries & Value-Based Decisioning

This notebook demonstrates:
1. Connecting to Snowflake `CUSTOMER_CHURN_DB` (with automatic local mock fallback)
2. Querying `RAW`, `ANALYTICS`, `ML`, and `REPORTING` schemas
3. Retention Decision Engine: Expected Retained Value, Intervention Costs, and Net Preserved ROI


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from snowflake.snowflake_client import SnowflakeClient
from snowflake.load_data import SnowflakeDataLoader

client = SnowflakeClient()
loader = SnowflakeDataLoader()
summary = loader.load_all()
print(f"Snowflake Mode: {summary['mode']}, Database: {summary['database']}")


### 1. Query Executive Churn Summary


In [ ]:
exec_summary = client.get_executive_summary()
pd.DataFrame([exec_summary])


### 2. Query High-Risk Customer Priorities


In [ ]:
q_high_risk = """
SELECT
    p.CUSTOMER_ID,
    p.CHURN_PROBABILITY,
    p.RISK_LEVEL,
    p.REVENUE_AT_RISK,
    p.ESTIMATED_CLV,
    r.RECOMMENDED_ACTION,
    r.URGENCY,
    r.ESTIMATED_NET_VALUE
FROM ML_CHURN_PREDICTIONS p
JOIN REPORTING_RETENTION_ACTIONS r ON p.CUSTOMER_ID = r.CUSTOMER_ID
WHERE p.RISK_LEVEL IN ('HIGH', 'CRITICAL')
ORDER BY p.REVENUE_AT_RISK DESC
LIMIT 10;
"""
high_risk_df = client.query_df(q_high_risk)
high_risk_df


### 3. Retention Campaign Portfolio ROI


In [ ]:
q_roi = """
SELECT
    RECOMMENDED_ACTION,
    URGENCY,
    COUNT(*) AS TOTAL_ACTIONS,
    ROUND(SUM(REVENUE_AT_RISK), 2) AS REVENUE_AT_RISK_COVERED,
    ROUND(SUM(ESTIMATED_INTERVENTION_COST), 2) AS TOTAL_BUDGET,
    ROUND(SUM(ESTIMATED_RETAINED_VALUE), 2) AS EXPECTED_RETURN,
    ROUND(SUM(ESTIMATED_NET_VALUE), 2) AS NET_PRESERVED_PROFIT
FROM REPORTING_RETENTION_ACTIONS
GROUP BY RECOMMENDED_ACTION, URGENCY
ORDER BY NET_PRESERVED_PROFIT DESC;
"""
roi_df = client.query_df(q_roi)
roi_df
